# 🎮 League of Legends Dataset Collection & Continuous Replay Pipeline
This notebook provides an automated, end-to-end scraper pipeline:
1. **Continuous Producer-Consumer Architecture**: Automatically discovers matches (e.g. Platinum/Emerald/Master) via the Riot Games API, queues `.rofl` downloads smoothly in the background, and continuously records minimap screenshots for model training.
2. **Champion Filter**: Automatically inspects every match before downloading/scraping to skip blacklisted champions (e.g. Shyvana, Fiddlesticks, Xin Zhao, Mel, Locke, Neeko).
3. **Overnight Reliability**: Automatic process recovery, dynamic auth discovery, and checkpointing.

### 1. Initialize Replay Scraper Engine

In [1]:
from autoLeague.replays.scraper import ReplayScraper
import os
from dotenv import load_dotenv, find_dotenv

# ── Load Riot API Key from .env ────────────────────────────────
load_dotenv(find_dotenv())
API_KEY = os.getenv("RIOT_API_KEY") or os.getenv("API_KEY", "")

if not API_KEY:
    print("[!] Warning: RIOT_API_KEY not found in .env! (Refresh at https://developer.riotgames.com/)")
else:
    print(f"[OK] Loaded Riot API Key from .env: {API_KEY[:10]}...")

# Local fast replay folder 
REPLAY_DIR = r"C:\Users\Aiden\Videos\League of Legends\Replays"
SAVE_DIR   = r"C:\Users\Aiden\Coding\python_ws\LeagueMapPredictor\Dataset"

# ── Verify game.cfg has EnableReplayApi=1 ──────────────────────
GAME_CFG = r"C:\Riot Games\League of Legends\Config\game.cfg"
if os.path.exists(GAME_CFG):
    with open(GAME_CFG, "r") as f:
        cfg = f.read()
    if "EnableReplayApi=1" not in cfg:
        if "[General]" in cfg:
            cfg = cfg.replace("[General]", "[General]\nEnableReplayApi=1", 1)
        else:
            cfg = "[General]\nEnableReplayApi=1\n" + cfg
        with open(GAME_CFG, "w") as f:
            f.write(cfg)
        print("⚠️  EnableReplayApi=1 was MISSING from game.cfg — auto-inserted!")
    else:
        print("✅ game.cfg: EnableReplayApi=1 confirmed")
else:
    print(f"⚠️  game.cfg not found at {GAME_CFG}")

scraper = ReplayScraper(
    game_dir=r"C:\Riot Games\League of Legends\Game",
    replay_dir=REPLAY_DIR,
    save_dir=SAVE_DIR,
    scraper_dir=r"C:\Users\Aiden\Coding\python_ws\LeagueMapPredictor\pyLoL\autoLeague\replays",
    lockfile_path=r"C:\Riot Games\League of Legends\lockfile",
    replay_speed=10,  # 10x replay playback speed
    region="SG2",
    monitor={"top": 546, "left": 1386, "width": 512, "height": 512}  # 512x512 minimap ROI
)

# List of champions to avoid in the dataset
BANNED_CHAMPIONS = [
    "Shyvana",
    "Fiddlesticks",
    "Fiddle",
    "XinZhao",
    "Xin Zhao",
    "Mel",
    "Locke",
    "Neeko",
    "Garen",
    "Ambessa",
    "Aurora",
    "Yunara",
    "Zaahen"
]

[OK] Loaded Riot API Key from .env: RGAPI-ccbe...
✅ game.cfg: EnableReplayApi=1 confirmed
📁 Replay Directory: 'C:\Users\Aiden\Videos\League of Legends\Replays' | Available .rofl files: 35


### 🔍 Step 1: Champion Inspection & Blacklist Test (Optional Verification)
Quickly check which champions are in any match before scraping.

In [2]:


# Check champion list for a sample match
sample_match = "SG2-171963955"
champs = scraper.get_match_champions_api(sample_match, API_KEY, routing_region="sea")
is_allowed, banned_found = scraper.check_banned_champions(champs, BANNED_CHAMPIONS)

print(f"Match {sample_match} Champions: {champs}")
if is_allowed:
    print("✅ ACCEPTED: Clean game, no banned champions.")
else:
    print(f"🚫 REJECTED: Contains banned champion(s): {banned_found}")

Match SG2-171963955 Champions: ['Irelia', 'Shaco', 'TwistedFate', 'Ezreal', 'Nautilus', 'Aatrox', 'Naafiri', 'Syndra', 'Kalista', 'Pyke']
✅ ACCEPTED: Clean game, no banned champions.


In [6]:
# ==============================================================================
# Delete all old .rofl files from previous update and get new ones
# ==============================================================================

# 1. Delete all old .rofl files from previous patch
deleted_count = scraper.clear_replays()
print(f"🗑️ Deleted {deleted_count} old .rofl file(s) from '{scraper.replay_dir}'")

# 2. Discover and download 3 fresh .rofl replays via Riot API & LCU
new_replays = scraper.fetch_and_download_replays(
    api_key=API_KEY,
    count=3,
    tier="PLATINUM",
    division="I",
    queue="RANKED_SOLO_5x5",
    routing_region="sea",
    banned_champions=BANNED_CHAMPIONS,
    days_back=5
)

current_rofls = [f for f in os.listdir(scraper.replay_dir) if f.endswith('.rofl')]
print(f"\n📁 Ready! {len(current_rofls)} replays available in folder: {current_rofls}")


🗑️ Deleted 0 old .rofl file(s) from 'C:\Users\Aiden\Videos\League of Legends\Replays'
🔍 Searching for 3 fresh replays in SG2 (PLATINUM I)...
⬇️ Downloading replay for SG2_178417574...
⚠️ Could not download SG2_178417574
🚫 Skipping SG2_178410756: contains banned champion(s): ['Yunara']
🚫 Skipping SG2_178422522: contains banned champion(s): ['Ambessa']
⬇️ Downloading replay for SG2_178294770...
⚠️ Could not download SG2_178294770
⬇️ Downloading replay for SG2_178899790...
✅ Downloaded: SG2-178899790.rofl
⬇️ Downloading replay for SG2_178860070...
✅ Downloaded: SG2-178860070.rofl
🚫 Skipping SG2_178851502: contains banned champion(s): ['Locke']
⬇️ Downloading replay for SG2_178841924...
✅ Downloaded: SG2-178841924.rofl
🎉 Successfully downloaded 3/3 replays.

📁 Ready! 3 replays available in folder: ['SG2-178841924.rofl', 'SG2-178860070.rofl', 'SG2-178899790.rofl']


### 🧪 Step 2: Single Match Test (Quick Verification)
Test that your League Client connects, applies camera/POV settings, and captures frames cleanly.

In [7]:
local_rofls = [f.replace(".rofl", "") for f in os.listdir(scraper.replay_dir) if f.endswith(".rofl")]
print(f"Available Replays ({len(local_rofls)}): {local_rofls[:5]}")

if local_rofls:
    test_match = local_rofls[-1]
    print(f"\n🎯 Testing scraper on: {test_match}")
    
    result = scraper.scrape_match(
        game_id=test_match,
        teams=("All", "Blue"),  # Scrapes All POV then Blue POV
        start_sec=1*60 - 5,     # 00:55 (Minion wave spawn)
        end_sec=25*60 - 5,      # 24:55 (Mid/Late game)
        speed=10,
        skip_existing=False     # Force scrape for verification
    )
    print("\nTest Result:", result)
else:
    print("❌ No .rofl files found in Replays folder.")

Available Replays (3): ['SG2-178841924', 'SG2-178860070', 'SG2-178899790']

🎯 Testing scraper on: SG2-178899790
🚀 Waiting for replay 178899790 (All POV) to load....... ✅ Connected!
📊 Saved player states (887 frames) → 'C:\Users\Aiden\Coding\python_ws\LeagueMapPredictor\Dataset\SG2-178899790\player_states.json'
📸 Captured 887 frames for SG2-178899790 (All POV)
🚀 Waiting for replay 178899790 (Blue POV) to load... ✅ Connected!
📊 Saved player states (909 frames) → 'C:\Users\Aiden\Coding\python_ws\LeagueMapPredictor\Dataset\SG2-178899790\player_states.json'
📸 Captured 911 frames for SG2-178899790 (Blue POV)

Test Result: {'status': 'success', 'game_id': 'SG2-178899790', 'frames': {'All': 887, 'Blue': 911}}


## 🌙 Step 3: Continuous Overnight Pipeline (Search ➔ Filter ➔ Download ➔ Scrape)

This runs the complete **Producer-Consumer pipeline**:
- **Downloader (Background)**: Searches Riot API for matches matching your desired tier (e.g. `PLATINUM`, `EMERALD`, `MASTER`), **filters out games with banned champions**, and downloads allowed `.rofl` files smoothly one-by-one.
- **Scraper (Main)**: Automatically picks up downloaded `.rofl` files, extracts `All` and `Blue` POVs, and saves them to `Dataset/`.
- **Runs Continuously** until your `TARGET_MATCHES` goal is reached.

In [3]:
# Configure your target dataset collection parameters
TARGET_MATCHES   = 40        # Goal match count (~70,000 frames)
TARGET_TIER      = "PLATINUM"  # 'PLATINUM', 'EMERALD', 'DIAMOND', or 'MASTER'
DIVISION         = "I"
SPEED            = 10          # 10x playback speed

# Start Continuous Pipeline
summary = scraper.run_continuous_pipeline(
    api_key=API_KEY,
    target_match_count=TARGET_MATCHES,
    tier=TARGET_TIER,
    division=DIVISION,
    queue="RANKED_SOLO_5x5",
    routing_region="sea",
    teams=("All", "Blue"),
    start_sec=1*60 - 5,            # 00:55
    end_sec=25*60 - 5,             # 24:55
    speed=SPEED,
    banned_champions=BANNED_CHAMPIONS, # Automatic champion filter
    delete_rofl_after_scrape=False,    # Set to True if you want to automatically free up disk space
    log_file="continuous_pipeline_summary.json"
)


🌟 CONTINUOUS SCRAPING PIPELINE | Goal: 40 Matches
Rank Filter: PLATINUM I | Queue: RANKED_SOLO_5x5 | Region: SG2 (sea)
POVs: ('All', 'Blue') | Speed: 10x | Auto-Cleanup ROFL: False
🚫 Champion Blacklist: ['Shyvana', 'Fiddlesticks', 'Fiddle', 'XinZhao', 'Xin Zhao', 'Mel', 'Locke', 'Neeko', 'Garen', 'Ambessa', 'Aurora', 'Yunara', 'Zaahen']



Scraped Matches:   0%|                                 | 0/40 [00:00<?, ?it/s, active=SG2-178640379]


🎬 Starting scrape for SG2-178640379...
🚀 Waiting for replay 178640379 (All POV) to load.... ✅ Connected!
📊 Saved player states (1357 frames) → 'C:\Users\Aiden\Coding\python_ws\LeagueMapPredictor\Dataset\SG2-178640379\player_states.json'
📸 Captured 1357 frames for SG2-178640379 (All POV)
🚀 Waiting for replay 178640379 (Blue POV) to load... ✅ Connected!


Scraped Matches:   2%|▌                     | 1/40 [05:38<3:40:02, 338.52s/it, active=SG2-178640379]

📊 Saved player states (1413 frames) → 'C:\Users\Aiden\Coding\python_ws\LeagueMapPredictor\Dataset\SG2-178640379\player_states.json'
📸 Captured 1406 frames for SG2-178640379 (Blue POV)
✅ Finished SG2-178640379 (1/40) | Frames: {'All': 1357, 'Blue': 1406}


...

Scraped Matches:  98%|████████████████████▍| 39/40 [3:30:10<05:21, 321.83s/it, active=SG2-178883676]


🎬 Starting scrape for SG2-178883676...
🚀 Waiting for replay 178883676 (All POV) to load... ✅ Connected!
📊 Saved player states (1407 frames) → 'C:\Users\Aiden\Coding\python_ws\LeagueMapPredictor\Dataset\SG2-178883676\player_states.json'
📸 Captured 1407 frames for SG2-178883676 (All POV)
🚀 Waiting for replay 178883676 (Blue POV) to load... ✅ Connected!


Scraped Matches: 100%|█████████████████████| 40/40 [3:35:43<00:00, 323.58s/it, active=SG2-178883676]

📊 Saved player states (1410 frames) → 'C:\Users\Aiden\Coding\python_ws\LeagueMapPredictor\Dataset\SG2-178883676\player_states.json'
📸 Captured 1399 frames for SG2-178883676 (Blue POV)
✅ Finished SG2-178883676 (40/40) | Frames: {'All': 1407, 'Blue': 1399}



🏁 CONTINUOUS PIPELINE FINISHED
🎯 Target: 40 | ✅ Successfully Scraped: 40
⏩ Skipped: 0 | 🚫 Filtered (Banned Champs/ARAM): 43
❌ Failed: 1
📄 Summary log: C:\Users\Aiden\Coding\python_ws\LeagueMapPredictor\Dataset\continuous_pipeline_summary.json
